In [5]:
from notebook_cache import NotebookCache

cache = NotebookCache("../cache/rl")

# RL on top of BC, cheaply (pick_place_cube)

Start from a policy that works sometimes, let it practise in the sim, keep what worked. Reward = the env's success flag at the end of an episode.

### step 0 — the starting policy: BC from `bc_policy.ipynb` (top + wrist, random shift, ~28%)

In [6]:
import os
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from torch import nn

from dream_robot.policies.bc.data import random_shift

REPO = Path.cwd().parent
dev = "cuda"
cams = ["top", "wrist"]

In [7]:
@cache
def load_data():
    ds = LeRobotDataset("dream_robot/pick_place_cube", root=REPO / "data/robosuite/pick_place_cube")
    state = np.asarray(ds.hf_dataset["observation.state"], dtype=np.float32)
    action = np.asarray(ds.hf_dataset["action"], dtype=np.float32)
    episode = np.asarray(ds.hf_dataset["episode_index"])
    ims = {c: np.zeros((len(ds), 3, 128, 128), dtype=np.uint8) for c in cams}
    for i in range(len(ds)):
        item = ds[i]
        for c in cams:
            ims[c][i] = (np.asarray(item[f"observation.images.{c}"]) * 255).round().astype(np.uint8)
    return state, action, episode, ims

load_data: not cached, will compute


In [8]:
state, action, episode, ims = load_data()
val_eps = np.random.default_rng(0).permutation(25)[:5]  # same split as bc_policy
tr = ~np.isin(episode, val_eps)
print(f"{state.shape=}  train frames {tr.sum()}")

state.shape=(5288, 8)  train frames 4218


In [9]:
# the demo data, on the GPU; normalisation stays fixed from here on
IM = {c: torch.tensor(ims[c][tr]).to(dev) for c in cams}
ST = torch.tensor(state[tr]).to(dev)
AC = torch.tensor(action[tr]).to(dev)

s_mean, s_std = ST.mean(0), ST.std(0).clamp_min(1e-3)
a_mean, a_std = AC.mean(0), AC.std(0).clamp_min(1e-3)
im_stats = {}
for c in cams:
    sub = IM[c][::20].float() / 255
    im_stats[c] = (sub.mean((0, 2, 3)).reshape(1, 3, 1, 1), sub.std((0, 2, 3)).reshape(1, 3, 1, 1))

In [10]:
class SpatialSoftmax(nn.Module):
    def __init__(self, h, w):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(()))
        v, u = torch.meshgrid(torch.linspace(-1, 1, h), torch.linspace(-1, 1, w), indexing="ij")
        self.register_buffer("grid_u", u.reshape(-1))
        self.register_buffer("grid_v", v.reshape(-1))

    def forward(self, f):  # [B, C, H, W] -> [B, 2C]
        B, C, H, W = f.shape
        a = torch.softmax(f.reshape(B * C, H * W) / self.log_t.exp(), -1)
        return torch.stack([a @ self.grid_u, a @ self.grid_v], -1).reshape(B, C * 2)


def make_encoder():
    return nn.Sequential(
        nn.Conv2d(3, 32, 5, 2, 2),
        nn.ReLU(),
        nn.Conv2d(32, 64, 3, 2, 1),
        nn.ReLU(),
        nn.Conv2d(64, 128, 3, 2, 1),
        nn.ReLU(),
        nn.Conv2d(128, 64, 3, 1, 1),
    )


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.encs = nn.ModuleDict({c: make_encoder() for c in cams})
        self.sss = nn.ModuleDict({c: SpatialSoftmax(16, 16) for c in cams})
        self.head = nn.Sequential(nn.Linear(128 * len(cams) + 8, 512), nn.ReLU(), nn.Linear(512, 8))

    def forward(self, ims, st):  # {cam: [B, 3, H, W] in [0, 1]}, [B, 8] -> [B, 8] in std units
        st = (st - s_mean) / s_std
        feats = [self.sss[c](self.encs[c]((ims[c] - im_stats[c][0]) / im_stats[c][1])) for c in cams]
        return self.head(torch.cat(feats + [st], -1))

In [39]:
def train(net, IM, ST, AC, steps=3000):
    # L1 in std units, random shift, as bc_policy step 10
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3)
    net.train()
    for step in range(steps + 1):
        i = torch.randint(0, len(ST), (64,), device=ST.device)  # data may sit on the CPU
        x = {c: random_shift(IM[c][i].to(dev).float() / 255, 4) for c in cams}
        loss = (net(x, ST[i].to(dev)) - (AC[i].to(dev) - a_mean) / a_std).abs().mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 1000 == 0:
            print(f"{step:5d}  train {loss.item():.4f}")
    return {k: v.cpu() for k, v in net.state_dict().items()}  # weights on the CPU for the cache

In [12]:
@cache
def train_bc(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    return train(Net().to(dev), IM, ST, AC)

train_bc: not cached, will compute


In [17]:
seed = 1  # seed 0 got 1/20
bc = Net().to(dev)
bc.load_state_dict(train_bc(seed))
bc.eval();

    0  train 0.8245
 1000  train 0.0651
 2000  train 0.0492
 3000  train 0.0275


In [18]:
os.environ.setdefault("MUJOCO_GL", "egl")
import logging

logging.disable(logging.WARNING)
import robosuite  # noqa: F401

logging.disable(logging.NOTSET)
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)

from dream_robot.core.registry import make_env

env = make_env("robosuite/pick_place_cube")


@torch.no_grad()
def act(net, obs):
    x = {c: torch.from_numpy(obs.images[c]).permute(2, 0, 1)[None].to(dev).float() / 255 for c in cams}
    a = (net(x, torch.from_numpy(obs.state)[None].to(dev)) * a_std + a_mean)[0].cpu().numpy()
    a[7] = np.clip(a[7], 0.0, 1.0)
    return a


def rollout(net, seed):
    obs = env.reset(seed=seed)
    for step in range(600):
        r = env.step(act(net, obs))
        obs = r.observation
        if r.terminated or r.truncated:
            break
    return r.success, str(r.failure_mode), step + 1


def evaluate(net, seeds=range(1000, 1020)):
    res = [rollout(net, s) for s in seeds]
    fails = Counter(m for ok, m, _ in res if not ok)
    print(f"{sum(ok for ok, _, _ in res)}/{len(res)}  failures {dict(fails)}")
    return res

In [19]:
@cache
def eval_bc(seed):  # seed: which trained bc this is, for the cache key
    return evaluate(bc)

eval_bc: not cached, will compute


In [20]:
res_bc = eval_bc(seed)

6/20  failures {'no_grasp': 14}


### step 1 — practice: 100 new cube positions, keep the successful episodes

Seeds 2000+, never the eval seeds 1000–1019.

In [21]:
@cache
def practice(seed, seeds=range(2000, 2100)):  # seed: which trained bc plays
    wins, modes = [], []
    for s in seeds:
        obs = env.reset(seed=s)
        ep = {c: [] for c in cams} | {"state": [], "action": []}
        for step in range(600):
            a = act(bc, obs)
            for c in cams:
                ep[c].append(obs.images[c].transpose(2, 0, 1))  # [3, H, W] uint8
            ep["state"].append(obs.state)
            ep["action"].append(a)
            r = env.step(a)
            obs = r.observation
            if r.terminated or r.truncated:
                break
        modes.append("success" if r.success else str(r.failure_mode))
        if r.success:
            wins.append({k: np.stack(v) for k, v in ep.items()})
    return wins, modes

practice: not cached, will compute


In [22]:
wins, modes = practice(seed)
print(f"{len(wins)}/{len(modes)} succeeded  {dict(Counter(modes))}")
print("frames kept:", sum(len(w["action"]) for w in wins), " (demos: 4218)")

21/100 succeeded  {'no_grasp': 78, 'success': 21, 'wrong_target': 1}
frames kept: 5540  (demos: 4218)


### step 2 — learn from the successes: keep training bc on demos + its 21 wins, then eval

In [23]:
@cache
def finetune_round1(seed):
    IM2 = {c: torch.cat([IM[c], torch.tensor(np.concatenate([w[c] for w in wins])).to(dev)]) for c in cams}
    ST2 = torch.cat([ST, torch.tensor(np.concatenate([w["state"] for w in wins])).to(dev)])
    AC2 = torch.cat([AC, torch.tensor(np.concatenate([w["action"] for w in wins])).to(dev)])
    np.random.seed(seed)
    torch.manual_seed(seed)
    net = Net().to(dev)
    net.load_state_dict(bc.state_dict())  # start from bc, not from scratch
    return train(net, IM2, ST2, AC2)

finetune_round1: not cached, will compute


In [24]:
rl1 = Net().to(dev)
rl1.load_state_dict(finetune_round1(seed))
rl1.eval();

    0  train 0.0278
 1000  train 0.0275
 2000  train 0.0213
 3000  train 0.0248


In [25]:
@cache
def eval_rl1(seed):
    return evaluate(rl1)

eval_rl1: not cached, will compute


In [26]:
res_rl1 = eval_rl1(seed)
print("bc was 6/20")

3/20  failures {'no_grasp': 16, 'dropped': 1}
bc was 6/20


### step 3 — 20 eval episodes can't tell 6 from 3; score both on 100 (seeds 1000–1099, ~5 min each)

In [27]:
nets = {"bc": bc, "rl1": rl1}

In [28]:
@cache
def eval100(name, seed):
    return evaluate(nets[name], range(1000, 1100))

eval100: not cached, will compute


In [29]:
for name in nets:
    print(name, end="  ")
    eval100(name, seed)

bc  23/100  failures {'no_grasp': 74, 'wrong_target': 3}
rl1  17/100  failures {'no_grasp': 78, 'dropped': 2, 'wrong_target': 3}


bc 23/100, rl1 17/100: round 1 bought nothing (gap ~1 std error, so maybe a small loss).

Why: bc is deterministic, so its 21 practice wins are positions it already solves. Copying them adds no new information about the 78 positions where it fails. Keeping successes only helps if practice sometimes succeeds where the policy usually fails, and that needs **exploration**.

### step 4 — round 2: practise with noise on every action, keep the wins, train, eval on 100

Noise = 0.2 × each action's std (arm ~0.01–0.04 rad). The wins keep the actions actually played, noise included.

In [30]:
@cache
def practice_noisy(seed, sigma=0.2, seeds=range(2000, 2100)):
    rng = np.random.default_rng(0)
    std = a_std.cpu().numpy()
    wins, modes = [], []
    for s in seeds:
        obs = env.reset(seed=s)
        ep = {c: [] for c in cams} | {"state": [], "action": []}
        for step in range(600):
            a = act(bc, obs) + sigma * std * rng.standard_normal(8).astype(np.float32)
            a[7] = np.clip(a[7], 0.0, 1.0)
            for c in cams:
                ep[c].append(obs.images[c].transpose(2, 0, 1))
            ep["state"].append(obs.state)
            ep["action"].append(a)
            r = env.step(a)
            obs = r.observation
            if r.terminated or r.truncated:
                break
        modes.append("success" if r.success else str(r.failure_mode))
        if r.success:
            wins.append({k: np.stack(v) for k, v in ep.items()} | {"seed": s})
    return wins, modes

practice_noisy: not cached, will compute


In [31]:
wins_n, modes_n = practice_noisy(seed)
new = sum(modes[w["seed"] - 2000] != "success" for w in wins_n)
print(f"noisy: {len(wins_n)}/100 succeeded  {dict(Counter(modes_n))}")
print(f"wins on positions the noise-free run failed: {new}   (noise-free: {len(wins)}/100)")

noisy: 21/100 succeeded  {'no_grasp': 73, 'wrong_target': 4, 'success': 21, 'dropped': 2}
wins on positions the noise-free run failed: 13   (noise-free: 21/100)


In [32]:
@cache
def finetune_round2(seed):
    IM2 = {c: torch.cat([IM[c], torch.tensor(np.concatenate([w[c] for w in wins_n])).to(dev)]) for c in cams}
    ST2 = torch.cat([ST, torch.tensor(np.concatenate([w["state"] for w in wins_n])).to(dev)])
    AC2 = torch.cat([AC, torch.tensor(np.concatenate([w["action"] for w in wins_n])).to(dev)])
    np.random.seed(seed)
    torch.manual_seed(seed)
    net = Net().to(dev)
    net.load_state_dict(bc.state_dict())
    return train(net, IM2, ST2, AC2)

finetune_round2: not cached, will compute


In [33]:
rl2 = Net().to(dev)
rl2.load_state_dict(finetune_round2(seed))
rl2.eval()
nets["rl2"] = rl2
eval100("rl2", seed)
print("bc 23/100, rl1 17/100")

    0  train 0.0986
 1000  train 0.1154
 2000  train 0.1170
 3000  train 0.1001
54/100  failures {'no_grasp': 39, 'wrong_target': 4, 'dropped': 3}
bc 23/100, rl1 17/100


```-
                practice wins   new positions   eval (100 seeds)
bc              —               —               23/100
rl1 (no noise)  21/100          0               17/100
rl2 (noise 0.2) 21/100          13              54/100
```

Same number of wins; what changed is that 13 of them are positions bc fails on, and every noisy episode is slightly off bc's usual path. Train loss sits at ~0.10 (vs 0.025): the noise itself can't be fitted, only its average.

### step 5 — more rounds, one at a time

Round k: rl(k-1) practises with noise on 100 fresh positions; train rl(k-1) on demos + the wins of every round so far; eval on 100.

In [34]:
def play_noisy(net, seeds, sigma=0.2):
    # as practice_noisy, for any net
    rng = np.random.default_rng(0)
    std = a_std.cpu().numpy()
    wins, modes = [], []
    for s in seeds:
        obs = env.reset(seed=s)
        ep = {c: [] for c in cams} | {"state": [], "action": []}
        for step in range(600):
            a = act(net, obs) + sigma * std * rng.standard_normal(8).astype(np.float32)
            a[7] = np.clip(a[7], 0.0, 1.0)
            for c in cams:
                ep[c].append(obs.images[c].transpose(2, 0, 1))
            ep["state"].append(obs.state)
            ep["action"].append(a)
            r = env.step(a)
            obs = r.observation
            if r.terminated or r.truncated:
                break
        modes.append("success" if r.success else str(r.failure_mode))
        if r.success:
            wins.append({k: np.stack(v) for k, v in ep.items()} | {"seed": s})
    return wins, modes


wins_by_round = {2: wins_n}

In [35]:
@cache
def practice_round(k, seed):
    first = 2000 + 100 * (k - 2)  # round 3 plays 2100-2199, round 4 2200-2299, ...
    return play_noisy(nets[f"rl{k - 1}"], range(first, first + 100))

practice_round: not cached, will compute


In [40]:
@cache
def finetune_round(k, seed):
    pool = [w for j in range(2, k + 1) for w in wins_by_round[j]]
    # on the CPU: demos + every round's wins outgrow the 8 GB GPU; train moves each batch over
    IM2 = {c: torch.cat([IM[c].cpu(), torch.tensor(np.concatenate([w[c] for w in pool]))]) for c in cams}
    ST2 = torch.cat([ST.cpu(), torch.tensor(np.concatenate([w["state"] for w in pool]))])
    AC2 = torch.cat([AC.cpu(), torch.tensor(np.concatenate([w["action"] for w in pool]))])
    np.random.seed(seed)
    torch.manual_seed(seed)
    net = Net().to(dev)
    net.load_state_dict(nets[f"rl{k - 1}"].state_dict())  # keep improving the last policy
    return train(net, IM2, ST2, AC2)

finetune_round: not cached, will compute


In [37]:
def one_round(k):
    wins_by_round[k], modes_k = practice_round(k, seed)
    print(f"round {k} practice: {len(wins_by_round[k])}/100  {dict(Counter(modes_k))}")
    net = Net().to(dev)
    net.load_state_dict(finetune_round(k, seed))
    nets[f"rl{k}"] = net.eval()
    print(f"rl{k}", end="  ")
    eval100(f"rl{k}", seed)

In [41]:
one_round(3)
print("bc 23/100, rl2 54/100")

round 3 practice: 50/100  {'success': 50, 'no_grasp': 46, 'dropped': 2, 'wrong_target': 2}
    0  train 0.1447
 1000  train 0.1388
 2000  train 0.1590
 3000  train 0.1428
rl3  6/100  failures {'no_grasp': 94}
bc 23/100, rl2 54/100


rl3 6/100: collapsed. Two variants, each changing one thing vs rl3 (same data: demos + rounds 2 and 3 wins):

```-
            starts from   training seed
rl3         rl2           1
A rl3_s2    rl2           2
B rl3_bc    bc            1
```

In [44]:
@cache
def finetune_from_bc(k, seed):
    # as finetune_round, but starting from bc every time
    pool = [w for j in range(2, k + 1) for w in wins_by_round[j]]
    IM2 = {c: torch.cat([IM[c].cpu(), torch.tensor(np.concatenate([w[c] for w in pool]))]) for c in cams}
    ST2 = torch.cat([ST.cpu(), torch.tensor(np.concatenate([w["state"] for w in pool]))])
    AC2 = torch.cat([AC.cpu(), torch.tensor(np.concatenate([w["action"] for w in pool]))])
    np.random.seed(seed)
    torch.manual_seed(seed)
    net = Net().to(dev)
    net.load_state_dict(bc.state_dict())
    return train(net, IM2, ST2, AC2)

finetune_from_bc: not cached, will compute


In [45]:
for name, weights in [("rl3_s2", lambda: finetune_round(3, 2)), ("rl3_bc", lambda: finetune_from_bc(3, seed))]:
    net = Net().to(dev)
    net.load_state_dict(weights())
    nets[name] = net.eval()
    print(name, end="  ")
    eval100(name, seed)
print("rl2 54/100, rl3 6/100")

    0  train 0.1325
 1000  train 0.1396
 2000  train 0.1429
 3000  train 0.1458
rl3_s2  39/100  failures {'no_grasp': 59, 'dropped': 2}
    0  train 0.1498
 1000  train 0.1368
 2000  train 0.1610
 3000  train 0.1436
rl3_bc  1/100  failures {'no_grasp': 89, 'wrong_target': 10}
rl2 54/100, rl3 6/100


```-
            starts from   seed   eval
rl3         rl2           1       6/100
A rl3_s2    rl2           2      39/100
B rl3_bc    bc            1       1/100
```

Starting from bc didn't save it (B 1/100), another seed did (A 39/100). With noisy labels the result swings 1 → 39 on the training seed alone, so one run per arm (rl2's 54 included) says little. Train loss never drops (~0.14) and lr stays at 1e-3, so the last step's weights are a random point along a noisy path.